# Частина 2: Аналіз споживання електроенергії
**Завдання:** Завантажити датасет `Individual Household Electric Power Consumption`. Здійснити очищення даних (Data Cleaning), сформувати вибірки та проаналізувати часові витрати за допомогою `timeit`.

In [9]:
import pandas as pd

print("Виконання..")
df_power = pd.read_csv('household_power_consumption.txt', sep=';', low_memory=False, na_values=['?'])

df_power = df_power.dropna()

df_power['Datetime'] = pd.to_datetime(df_power['Date'] + ' ' + df_power['Time'], format='%d/%m/%Y %H:%M:%S')

print("Дані успішно очищено")
display(df_power.head())

Виконання..
Дані успішно очищено


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


### 3. Формування вибірок (Частина 1)
**Завдання:** 1. Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.
2. Обрати записи, у яких сила струму 19-20 А, і пральна машина та холодильник (група 2) споживають більше, ніж бойлер та кондиціонер (група 3).
Провести профілювання часу виконання.

In [10]:
cols_to_convert = ['Global_active_power', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df_power[cols_to_convert] = df_power[cols_to_convert].apply(pd.to_numeric, errors='coerce')
df_power = df_power.dropna()

def filter_high_power(df):
    """Повертає всі записи, де загальна активна потужність > 5 кВт"""
    return df[df['Global_active_power'] > 5.0]

print("Вибірка 1")
res1 = filter_high_power(df_power)
display(res1.head(3)) 

print("Час виконання Вибірки 1:")
%timeit -n 3 filter_high_power(df_power)


def filter_laundry_over_ac(df):
    """Сила струму 19-20 А, Sub_metering_2 (пралка) > Sub_metering_3 (бойлер)"""
    cond1 = df['Global_intensity'].between(19, 20)
    cond2 = df['Sub_metering_2'] > df['Sub_metering_3']
    return df[cond1 & cond2]

print("Вибірка 2")
res2 = filter_laundry_over_ac(df_power)
display(res2.head(3))

print("Час виконання Вибірки 2:")
%timeit -n 3 filter_laundry_over_ac(df_power)

Вибірка 1


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00


Час виконання Вибірки 1:
5.57 ms ± 157 μs per loop (mean ± std. dev. of 7 runs, 3 loops each)
Вибірка 2


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00


Час виконання Вибірки 2:
11.9 ms ± 730 μs per loop (mean ± std. dev. of 7 runs, 3 loops each)


### 4. Формування вибірок (Частина 2)
**Завдання:**
1. Обрати випадковим чином 500000 записів (без повторів) та обчислити середні величини 3-х груп споживання.
2. Обрати записи після 18:00 з потужністю > 6 кВт/хв, де група 2 є найбільшою. Обрати кожен 3-й результат з першої половини та кожен 4-й з другої.

In [11]:
def sample_and_average(df):
    sample_df = df.sample(n=500000, replace=False, random_state=42)
    return sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

print("Результат Вибірки 3 (Середні значення)")
res3 = sample_and_average(df_power)
print(res3)

print("Час виконання Вибірки 3:")
%timeit -n 3 sample_and_average(df_power)


def complex_evening_filter(df):    
    cond1 = df['Datetime'].dt.hour >= 18
    cond2 = df['Global_active_power'] > 6.0
    
    cond3 = (df['Sub_metering_2'] > df['Sub_metering_1']) & (df['Sub_metering_2'] > df['Sub_metering_3'])
    
    filtered_df = df[cond1 & cond2 & cond3].reset_index(drop=True)
    
    half_index = len(filtered_df) // 2
    first_half = filtered_df.iloc[:half_index]
    second_half = filtered_df.iloc[half_index:]
    
    part1 = first_half.iloc[::3]
    part2 = second_half.iloc[::4]
    
    return pd.concat([part1, part2])

print("Результат Вибірки 4")
res4 = complex_evening_filter(df_power)
display(res4.head(10)) 

print("Час виконання Вибірки 4:")
%timeit -n 3 complex_evening_filter(df_power)

Результат Вибірки 3 (Середні значення)
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64
Час виконання Вибірки 3:
170 ms ± 6.68 ms per loop (mean ± std. dev. of 7 runs, 3 loops each)
Результат Вибірки 4


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
3,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
6,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
9,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
12,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00
15,28/12/2006,21:08:00,7.352,0.000,235.45,31.2,1.0,73.0,17.0,2006-12-28 21:08:00
18,28/12/2006,21:11:00,9.048,0.000,231.48,39.0,34.0,71.0,16.0,2006-12-28 21:11:00
21,28/12/2006,21:14:00,9.118,0.108,231.18,39.4,36.0,72.0,16.0,2006-12-28 21:14:00
24,28/12/2006,21:17:00,7.040,0.130,233.27,30.2,37.0,38.0,17.0,2006-12-28 21:17:00
27,29/12/2006,21:16:00,6.146,0.116,230.53,26.6,0.0,70.0,0.0,2006-12-29 21:16:00


Час виконання Вибірки 4:
51.9 ms ± 1.6 ms per loop (mean ± std. dev. of 7 runs, 3 loops each)


### 5. Нормування, Стандартизація, Кореляція та One Hot Encoding
**Завдання:** 1. Пронормувати та стандартизувати вибраний датасет.
2. Підрахувати коефіцієнт Пірсона та Спірмена.
3. Провести One Hot Encoding категоріального атрибута.

In [13]:
df_math = res4.copy()

col = 'Global_active_power'

df_math['Power_Normalized'] = (df_math[col] - df_math[col].min()) / (df_math[col].max() - df_math[col].min())
df_math['Power_Standardized'] = (df_math[col] - df_math[col].mean()) / df_math[col].std()

print("1. Результат Нормування та Стандартизації")
display(df_math[[col, 'Power_Normalized', 'Power_Standardized']].head())


pearson_corr = df_math['Global_active_power'].corr(df_math['Global_intensity'], method='pearson')
spearman_corr = df_math['Global_active_power'].corr(df_math['Global_intensity'], method='spearman')

print("\n 2. Кореляція")
print(f"Коефіцієнт Пірсона: {pearson_corr:.4f}")
print(f"Коефіцієнт Спірмена: {spearman_corr:.4f}")


df_math['Day_of_Week'] = df_math['Datetime'].dt.day_name()

df_encoded = pd.get_dummies(df_math, columns=['Day_of_Week'], dtype=int)

print("\n 3. Результат One Hot Encoding")
display(df_encoded.filter(regex='Datetime|Day_of_Week').head())

1. Результат Нормування та Стандартизації


,Global_active_power,Power_Normalized,Power_Standardized
0,6.052,0.010331,-1.028627
3,6.308,0.065433,-0.707237
6,6.386,0.082221,-0.609313
9,8.088,0.448558,1.527431
12,7.230,0.263883,0.450271



 2. Кореляція
Коефіцієнт Пірсона: 0.9943
Коефіцієнт Спірмена: 0.9840

 3. Результат One Hot Encoding


,Datetime,Day_of_Week_Friday,Day_of_Week_Monday,Day_of_Week_Saturday,Day_of_Week_Sunday,Day_of_Week_Thursday,Day_of_Week_Tuesday,Day_of_Week_Wednesday
0,2006-12-16 18:05:00,0,0,1,0,0,0,0
3,2006-12-16 18:08:00,0,0,1,0,0,0,0
6,2006-12-28 20:58:00,0,0,0,0,1,0,0
9,2006-12-28 21:02:00,0,0,0,0,1,0,0
12,2006-12-28 21:05:00,0,0,0,0,1,0,0
